# Fee Tier Acceleration Experiment — Analysis

**Hypothesis:** If we show institutional clients their progress toward the next fee tier (progress bar + monthly savings email), they'll increase trading volume to cross the threshold.

**Challenge:** Volume ↑ is good, but clients who cross tiers pay LOWER rates. Does the volume gain outweigh the margin compression?

**This notebook covers:**
1. Randomization check (balance table)
2. Average Treatment Effect (ATE) with three statistical tests
3. CUPED variance reduction
4. Revenue decomposition (the waterfall)
5. Heterogeneous Treatment Effects by distance-to-tier
6. Durability (effect decay over weeks)
7. Cannibalization check
8. Guardrail checks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

from analysis.statistical_tests import (
    compute_ate, run_all_tests, cuped_comparison, compute_balance_table
)
from analysis.revenue_decomposition import decompose_experiment, summarize_by_group
from analysis.hte import compute_hte, compute_hte_revenue
from analysis.durability import compute_weekly_effects, estimate_decay_rate, annualization_scenarios
from analysis.cannibalization import check_cannibalization
from validation.data_quality_checks import run_all_checks

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Load data
data_dir = Path('..') / 'data'
df = pd.read_csv(data_dir / 'processed' / 'experiment_summary.csv')
pre_activity = pd.read_csv(data_dir / 'raw' / 'pre_period_activity.csv')
post_activity = pd.read_csv(data_dir / 'raw' / 'post_period_activity.csv')
assignment = pd.read_csv(data_dir / 'raw' / 'assignment.csv')

print(f'Experiment table: {len(df)} clients')
print(f'Treatment: {(df["group"]=="treatment").sum()}, Control: {(df["group"]=="control").sum()}')

In [ ]:
# Step 0: Run data quality checks BEFORE any analysis
run_all_checks(df, pre_activity, assignment)

## 1. Randomization Check
Before looking at results, verify the groups are balanced.

In [ ]:
balance = compute_balance_table(
    df,
    covariates=['pre_volume', 'aum', 'tenure_months', 'n_products', 'pre_logins'],
)
balance

## 2. Average Treatment Effect
Three independent tests for robustness.

In [ ]:
t_vol = df[df['group'] == 'treatment']['post_volume'].values
c_vol = df[df['group'] == 'control']['post_volume'].values

ate = compute_ate(t_vol, c_vol)
print(f"ATE (volume): ${ate['ate']:,.0f} ({ate['pct_lift']:.1f}% lift)")
print(f"95% CI: [${ate['ci_lower']:,.0f}, ${ate['ci_upper']:,.0f}]")

tests = run_all_tests(t_vol, c_vol)
tests

## 3. CUPED Variance Reduction

In [ ]:
t_pre = df[df['group'] == 'treatment']['pre_volume'].values
c_pre = df[df['group'] == 'control']['pre_volume'].values

cuped = cuped_comparison(t_vol, c_vol, t_pre, c_pre)
print(f"Raw ATE:  ${cuped['raw_ate']:,.0f} ± ${cuped['raw_se']*1.96:,.0f}")
print(f"CUPED ATE: ${cuped['cuped_ate']:,.0f} ± ${cuped['cuped_se']*1.96:,.0f}")
print(f"Variance reduced by: {cuped['variance_reduction_pct']:.0f}%")

# TODO: Create side-by-side CI visualization (the CUPED money chart)

## 4. Revenue Decomposition
The core tension: Volume ↑ but Fee Rate ↓. What's the net?

In [ ]:
# Overall decomposition (treatment group only)
treatment_df = df[df['group'] == 'treatment']
print(f"Mean volume effect:      ${treatment_df['volume_effect'].mean():,.0f}")
print(f"Mean rate effect:        ${treatment_df['rate_effect'].mean():,.0f}")
print(f"Mean interaction effect: ${treatment_df['interaction_effect'].mean():,.0f}")
print(f"Mean net ΔRevenue:       ${treatment_df['revenue_change'].mean():,.0f}")

# By distance bucket
revenue_by_distance = summarize_by_group(df, 'distance_bucket')
revenue_by_distance

# TODO: Build waterfall chart (THE hero visual)

## 5. Heterogeneous Treatment Effects
Who benefits most? Who should get this feature?

In [ ]:
# HTE by distance to tier (the key cut)
hte_distance = compute_hte_revenue(df, 'distance_bucket')
hte_distance

# TODO: Forest plot visualization

In [ ]:
# HTE by segment
hte_segment = compute_hte_revenue(df, 'segment')
hte_segment

## 6. Durability (Effect Over Time)

In [ ]:
weekly = compute_weekly_effects(post_activity, assignment)
decay = estimate_decay_rate(weekly)
print(f"Initial effect: {decay['week_1_effect']:.1f}%")
print(f"Week 13 effect: {decay['week_13_effect']:.1f}%")
print(f"Weekly decay: {decay['weekly_decay_pct']:.2f}%")
print(f"Interpretation: {decay['interpretation']}")

# TODO: Line chart of weekly effects

## 7. Cannibalization Check

In [ ]:
cannibal = check_cannibalization(pre_activity, post_activity, assignment)
print(f"Control volume change: {cannibal['control_change_pct']:.1f}%")
print(f"p-value: {cannibal['p_value']:.4f}")
print(cannibal['interpretation'])

## 8. Guardrail Checks

In [ ]:
# Support tickets: should not spike
t_tickets = df[df['group'] == 'treatment']['ticket_change'].mean()
c_tickets = df[df['group'] == 'control']['ticket_change'].mean()
print(f"Ticket change — Treatment: {t_tickets:.1f}, Control: {c_tickets:.1f}")

# Login frequency: should not drop
t_logins = df[df['group'] == 'treatment']['login_change'].mean()
c_logins = df[df['group'] == 'control']['login_change'].mean()
print(f"Login change — Treatment: {t_logins:.1f}, Control: {c_logins:.1f}")

# Enterprise churn: must be 0
enterprise_treatment = df[(df['group'] == 'treatment') & (df['current_tier'] == 'Enterprise')]
print(f"Enterprise treatment churned: {enterprise_treatment['churned_6mo'].sum()}")